In [ ]:
import pandas as pd
df = pd.read_csv("/content/jan to may police violation_anonymized791b166.csv")
print(df.shape)
print(df.head(3))

(298450, 24)
           id   latitude  longitude  \
0  FKID000000  12.925557  77.618665   
1  FKID000001  12.905463  77.700778   
2  FKID000002  12.925449  77.618504   

                                            location vehicle_number  \
0  18th Main Road, Block 2, Koramangala, Bengalur...    FKN00GL0000   
1  Sarjapura Main Road, The Grove, Janatha Colony...    FKN00GL0001   
2  Koramangala 2nd Block, Kormangala West, Bengal...    FKN00GL0002   

  vehicle_type  description                                  violation_type  \
0          CAR          NaN  ["WRONG PARKING","PARKING NEAR ROAD CROSSING"]   
1          CAR          NaN                                  ["NO PARKING"]   
2          CAR          NaN      ["WRONG PARKING","PARKING IN A MAIN ROAD"]   

  offence_code        created_datetime  ...  center_code police_station  \
0    [112,104]  2023-11-20 00:28:46+00  ...          9.0       Madiwala   
1        [113]  2023-11-24 22:46:46+00  ...         82.0      Bellandur   
2  

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import json
import folium
from folium.plugins import HeatMap
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── LOAD ──────────────────────────────────────────────────
df = pd.read_csv("/content/jan to may police violation_anonymized791b166.csv")
print(f"Loaded: {df.shape}")

# ── VIOLATION PARSING ─────────────────────────────────────
def parse_violation(val):
    try:
        parsed = json.loads(val)
        return parsed if isinstance(parsed, list) else [val]
    except:
        return [str(val)]

df['violation_list']    = df['violation_type'].apply(parse_violation)
df['primary_violation'] = df['violation_list'].apply(lambda x: x[0] if x else 'UNKNOWN')
df['violation_count']   = df['violation_list'].apply(len)

# ── DATETIME ──────────────────────────────────────────────
df['created_datetime'] = pd.to_datetime(df['created_datetime'], format='mixed', utc=True)
df['hour_IST']         = (df['created_datetime'].dt.hour + 5) % 24
df['day_of_week']      = df['created_datetime'].dt.day_name()
df['month']            = df['created_datetime'].dt.month_name()

# ── JUNCTION ──────────────────────────────────────────────
df['junction_name'] = df['junction_name'].fillna('No Junction')
df['at_junction']   = df['junction_name'] != 'No Junction'

# ── GAP 2 FIX: ZONE TAGGING ───────────────────────────────
metro_keywords = [
    'metro', 'namma metro', 'railway station',
    'bus stand', 'bus station', 'terminus'
]
commercial_keywords = [
    'mall', 'market', 'plaza', 'commercial',
    'bazaar', 'shopping', 'complex', 'hub',
    'centre', 'center', 'brigade', 'mg road',
    'commercial street', 'sp road', 'avenue'
]

loc = df['location'].fillna('').str.lower()
df['near_metro']      = loc.str.contains('|'.join(metro_keywords))
df['near_commercial'] = loc.str.contains('|'.join(commercial_keywords))
df['zone_type'] = 'General'
df.loc[df['near_metro'],      'zone_type'] = 'Metro Zone'
df.loc[df['near_commercial'], 'zone_type'] = 'Commercial Zone'

# Fix 1 — also check junction name for metro/commercial
junc = df['junction_name'].fillna('').str.lower()
df['near_metro'] = df['near_metro'] | junc.str.contains('metro|railway|station')
df['near_commercial'] = df['near_commercial'] | junc.str.contains('market|plaza|mall|commercial')
df.loc[df['near_metro'], 'zone_type'] = 'Metro Zone'
df.loc[df['near_commercial'], 'zone_type'] = 'Commercial Zone'

print(f"Metro violations:      {df['near_metro'].sum():,}")
print(f"Commercial violations: {df['near_commercial'].sum():,}")

# ── VEHICLE SEVERITY ──────────────────────────────────────
severity_map = {
    'PRIVATE BUS': 5, 'BUS': 5,
    'LGV': 4, 'HGV': 4,
    'MAXI-CAB': 3, 'VAN': 3, 'GOODS AUTO': 3,
    'CAR': 2, 'PASSENGER AUTO': 2,
    'SCOOTER': 1, 'MOTOR CYCLE': 1, 'MOPED': 1
}
df['vehicle_severity'] = df['vehicle_type'].map(severity_map).fillna(2)
df['validated']        = df['validation_status'] == 'approved'

# ── DBSCAN ON SAMPLE (RAM safe) ───────────────────────────
df_geo = df.dropna(subset=['latitude', 'longitude']).copy()
coords_rad = np.radians(df_geo[['latitude', 'longitude']].values)
db = DBSCAN(eps=0.5/6371, algorithm='ball_tree',
            metric='haversine', min_samples=50).fit(coords_rad)
df_geo['cluster'] = db.labels_

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f"Hotspot clusters: {n_clusters}")
print(f"Clustered points: {(db.labels_ != -1).sum():,}")

# ── GAP 3 FIX: ROAD CAPACITY IMPACT ──────────────────────
# Lane blockage per vehicle type
capacity_map = {5: 2.0, 4: 1.5, 3: 1.2, 2: 1.0, 1: 0.5}
df_geo['lane_impact'] = df_geo['vehicle_severity'].map(capacity_map).fillna(1.0)

junction_impact = df_geo[df_geo['at_junction']].groupby('junction_name').agg(
    total_violations  = ('id', 'count'),
    lat               = ('latitude', 'mean'),
    lon               = ('longitude', 'mean'),
    avg_severity      = ('vehicle_severity', 'mean'),
    total_lane_impact = ('lane_impact', 'sum'),
    zone_type         = ('zone_type', lambda x: x.mode()[0])
).reset_index()

# Capacity reduction: assume 4 usable lanes per junction
# Each violation blocks lane_impact lanes for avg 8 mins per hour
junction_impact['capacity_reduction_pct'] = (
    (junction_impact['total_lane_impact'] /
     (junction_impact['total_violations'] * 4)) * 100
).clip(upper=95).round(1)

junction_impact = junction_impact.sort_values('total_violations', ascending=False)
print("\nTop 10 Junctions - Capacity Impact:")
print(junction_impact[['junction_name', 'total_violations',
                        'capacity_reduction_pct', 'zone_type']].head(10).to_string(index=False))

# ── CONGESTION SCORE ──────────────────────────────────────
def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

station_score = df.groupby('police_station').agg(
    total_violations      = ('id', 'count'),
    junction_violations   = ('at_junction', 'sum'),
    avg_severity          = ('vehicle_severity', 'mean'),
    multi_violations      = ('violation_count', lambda x: (x > 1).sum()),
    enforcement_gap       = ('validated', lambda x: (~x).sum()),
    metro_violations      = ('near_metro', 'sum'),
    commercial_violations = ('near_commercial', 'sum')
).reset_index()

station_score['congestion_score'] = (
    normalize(station_score['total_violations'])       * 0.28 +
    normalize(station_score['junction_violations'])    * 0.28 +
    normalize(station_score['avg_severity'])           * 0.15 +
    normalize(station_score['multi_violations'])       * 0.10 +
    normalize(station_score['enforcement_gap'])        * 0.10 +
    normalize(station_score['metro_violations'])       * 0.05 +
    normalize(station_score['commercial_violations'])  * 0.04
) * 100

station_score['congestion_score'] = station_score['congestion_score'].round(1)
station_score = station_score.sort_values('congestion_score', ascending=False)

print("\nTop 10 Stations by Congestion Score:")
print(station_score[['police_station', 'total_violations',
                      'junction_violations', 'congestion_score']].head(10).to_string(index=False))

# ── ENFORCEMENT RECOMMENDATION ────────────────────────────
def get_recommendation(station_name):
    row = station_score[station_score['police_station'] == station_name]
    if row.empty:
        return "Station not found"

    score      = row['congestion_score'].values[0]
    gap        = row['enforcement_gap'].values[0]
    station_df = df[df['police_station'] == station_name]
    peak_hour  = station_df['hour_IST'].mode()[0]
    peak_day   = station_df['day_of_week'].mode()[0]
    top_viol   = station_df['primary_violation'].mode()[0]
    top_zone   = station_df['zone_type'].mode()[0]

    if score >= 60:   units, priority = 5, "CRITICAL"
    elif score >= 40: units, priority = 3, "HIGH"
    elif score >= 20: units, priority = 2, "MEDIUM"
    else:             units, priority = 1, "LOW"

    return {
        'station':                  station_name,
        'priority':                 priority,
        'congestion_score':         score,
        'recommended_patrol_units': units,
        'deploy_at_hour_IST':       f"{peak_hour}:00 - {peak_hour+2}:00",
        'peak_day':                 peak_day,
        'top_violation_type':       top_viol,
        'hotspot_zone':             top_zone,
        'unresolved_violations':    int(gap)
    }

for s in ['Upparpet', 'Shivajinagar', 'City Market']:
    print(get_recommendation(s))
    print()

# ── GAP 1 FIX: HEATMAP ────────────────────────────────────
print("Building heatmap...")
heat_data = df_geo[['latitude', 'longitude', 'vehicle_severity']].dropna()

m = folium.Map(
    location=[12.9716, 77.5946],  # Bengaluru center
    zoom_start=12,
    tiles='CartoDB dark_matter'
)

# Layer 1: Violation heatmap
HeatMap(
    heat_data[['latitude', 'longitude', 'vehicle_severity']].values.tolist(),
    radius=10,
    blur=15,
    max_zoom=13,
    name='Violation Heatmap',
    gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'orange', 1.0: 'red'}
).add_to(m)

# Layer 2: Top junction markers
for _, row in junction_impact.head(20).iterrows():
    color = 'red' if row['capacity_reduction_pct'] > 60 else \
            'orange' if row['capacity_reduction_pct'] > 30 else 'green'
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=8,
        color=color,
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{row['junction_name']}</b><br>"
            f"Violations: {row['total_violations']:,}<br>"
            f"Capacity Reduction: {row['capacity_reduction_pct']}%<br>"
            f"Zone: {row['zone_type']}",
            max_width=250
        )
    ).add_to(m)

folium.LayerControl().add_to(m)
m.save("/content/parking_heatmap.html")
print("Heatmap saved.")

# ── SAVE ALL FILES ────────────────────────────────────────
df_geo.to_csv("/content/cleaned_violations.csv", index=False)
station_score.to_csv("/content/station_scores.csv", index=False)
junction_impact.to_csv("/content/junction_summary.csv", index=False)
print("All files saved. Pipeline complete.")

Loaded: (298450, 24)
Metro violations:      7,208
Commercial violations: 51,557
